This notebook puts the generated data necessary for figures into the figure_data folder

In [ ]:
import polars as pl
from pathlib import Path
from spcg import cluster_bootstrap_ci
import pandas as pd
import numpy as np
from scipy import sparse
import spcg.null_model as nm
import xgi

In [ ]:
RUN_DIR = Path(
    "../output/run_20260709_053144_directed"
)  # this run's figures/tables + manifest
DATA_DIR = Path("../output/study_data")  # cached data tables (pairs, hops, semantic)
FIGURE_DATA_DIR = Path("../figure_data")
SEED = 16
METHOD = "cluster_average"
MODEL_SAFE = "all-MiniLM-L6-v2"
MAX_SHARED = 8
NULL_STATISTIC = "interaction"

S_EFF_CAP = 25

fname = DATA_DIR / f"cosine_overlap_maxusers_pairs_seed{SEED}.parquet"
df_pairs = pd.read_parquet(fname)

In [ ]:
def build_agg_count(pair_df, seed=SEED, max_shared=MAX_SHARED, n_boot=500, save=True):
    """Stage-1 agg (cosine vs raw shared-pack count): use the pipeline's saved
    `cosine_overlap_maxusers_agg_seed{seed}` in RUN_DIR if present, else rebuild
    it from the cached per-pair table in DATA_DIR via cluster_bootstrap_ci --
    the identical aggregation the pipeline runs (deterministic given `seed`)."""
    need = {"user_a", "user_b", "shared_packs", "cosine"}
    miss = need - set(pair_df.columns)
    if miss:
        raise KeyError(f"per-pair table is missing column(s): {sorted(miss)}")
    print(f"  loaded {len(pair_df):,} pairs; aggregating (n_boot={n_boot}) ...")
    agg = cluster_bootstrap_ci(
        pair_df,
        value_col="cosine",
        group_col="shared_packs",
        n_boot=n_boot,
        seed=seed,
        max_shared=max_shared,
    )
    return agg

In [ ]:
df = build_agg_count(df_pairs, seed=SEED, max_shared=8, n_boot=500, save=True)
df = df.rename(columns={"shared_packs": "s_raw", "mean": "mean_cosine"})
df.to_csv(
    FIGURE_DATA_DIR / f"similarity_vs_raw_overlap_max8_seed{SEED}.csv", index=False
)

df = build_agg_count(df_pairs, seed=SEED, max_shared=S_EFF_CAP, n_boot=500, save=True)
df = df.rename(columns={"shared_packs": "s_raw", "mean": "mean_cosine"})
df.to_csv(
    FIGURE_DATA_DIR / f"similarity_vs_raw_overlap_max{S_EFF_CAP}_seed{SEED}.csv",
    index=False,
)

In [ ]:
df = pd.read_parquet(
    DATA_DIR / f"semantic_overlap_maxusers_{MODEL_SAFE}_seed{SEED}.parquet"
)
# processing: continuous n_eff vs cosine from the augmented pair table (+ shared trend)
INCLUDE_ZERO = False  # include the n_eff==0 disjoint-baseline pairs?

df = df.dropna()
if not INCLUDE_ZERO:
    df = df[df["n_eff"] > 0]

anonymized_df = df[["shared_packs", "n_eff", "cosine"]].copy()
anonymized_df.rename(
    columns={"shared_packs": "s_raw", "n_eff": "s_eff", "cosine": "cosine"}
).to_csv(
    FIGURE_DATA_DIR / f"similarity_vs_renormalized_overlap_seed{SEED}.csv", index=False
)

In [ ]:
df_null = pd.read_parquet("../output/paper_numbers/s7_degree_matched_curve.parquet")
df_null = df_null[["shared_packs", "n_obs", "matched_zero_mean_cosine"]].copy()
df_null.rename(
    columns={"shared_packs": "s_raw", "n_obs": "n_obs", "matched_zero_mean_cosine": "mean_cosine"}
).to_csv(FIGURE_DATA_DIR / f"similarity_vs_overlap_null_seed{SEED}.csv", index=False)

In [ ]:
df = pd.read_csv("../output/paper_numbers/s3_cell_counts.csv")
df.replace(
    to_replace={
        "n_eff_stratum": {
            "n_eff=0": 0,
            "n_eff[1,3]": 1,
            "n_eff>3": 2,
        },
        "hops": {
            "far/unreachable": 4,
        },
    },
    inplace=True,
)
df = df.rename(
    columns={
        "n_eff_stratum": "s_eff_regime",
        "hops": "network_distance",
    }
)
df.to_csv(
    FIGURE_DATA_DIR
    / f"similarity_vs_renormalized_overlap_and_network_distance_seed{SEED}.csv",
    index=False,
)

In [ ]:
PACKS_PATH = "../starterpack_hif.json" 

N_NULL_REPS = 5

H = xgi.read_hif(PACKS_PATH)

In [ ]:
degrees = H.nodes.degree.asdict()
degrees_a = [min(degrees[i], degrees[j]) for i, j in df_pairs[["user_a", "user_b"]].to_numpy()] 
degrees_b = [max(degrees[i], degrees[j]) for i, j in df_pairs[["user_a", "user_b"]].to_numpy()]

df_pairs["k_u"] = degrees_a
df_pairs["k_v"] = degrees_b
df_pairs[["k_u", "k_v", "cosine"]].to_csv(FIGURE_DATA_DIR / f"similarity_vs_degrees_seed{SEED}.csv", index=False)

In [ ]:
pairs = pd.read_parquet(DATA_DIR / "directed" / f"pairs_with_hops_n_eff_seed{SEED}.parquet")
p0 = pairs[pairs.shared_packs == 0]
p1 = pairs[pairs.shared_packs >= 1]

In [ ]:
deg_bin0 = np.concatenate([H.nodes(p0.user_a).degree.asnumpy(), H.nodes(p0.user_b).degree.asnumpy()])   # (1) uniform s=0
deg_bin1 = np.concatenate([H.nodes(p1.user_a).degree.asnumpy(), H.nodes(p1.user_b).degree.asnumpy()])   # (2) s>=1 pairs

In [ ]:
PACKS_PATH = "../starterpacks.jsonl"   # see run_paper_numbers.sh
N_NULL_REPS = 5

# pack degree k_u for every user, straight from the incidence margins.
B, user_ids, _pack_ids = nm.load_incidence(PACKS_PATH)
d_u, _k_e = nm.margins(B)                       # d_u[i] = #packs user i belongs to
row_of = {u: i for i, u in enumerate(user_ids)}

def _deg_of(dids):
    idx = [row_of[u] for u in dids if u in row_of]
    return d_u[np.asarray(idx, dtype=int)]

# populations 1 & 2 from the realized pair table (endpoints NOT deduped, so
# population 2 keeps its degree weighting).
pairs = pd.read_parquet(DATA_DIR / "directed" / f"pairs_with_hops_n_eff_seed{SEED}.parquet")
p0 = pairs[pairs.shared_packs == 0]
p1 = pairs[pairs.shared_packs >= 1]
deg_bin0 = np.concatenate([_deg_of(p0.user_a), _deg_of(p0.user_b)])   # (1) uniform s=0
deg_bin1 = np.concatenate([_deg_of(p1.user_a), _deg_of(p1.user_b)])   # (2) s>=1 pairs

# population 3: users appearing in co-member pairs under BiCM replicates.
# candidate pool matches census_comember_pairs (users in >=2 packs), and each
# co-member pair contributes both endpoints — same weighting as population 2.
cand = np.flatnonzero(d_u >= 2)
x, y = nm.bicm_fit(B)
parts = []
for rep in range(N_NULL_REPS):
    B_rand = nm.bicm_sample(x, y, seed=SEED + rep, rows=cand, pbar=True)
    S = sparse.triu(B_rand @ B_rand.T, k=1).tocoo()
    keep = S.data >= 1
    parts.append(d_u[cand[S.row[keep]]])
    parts.append(d_u[cand[S.col[keep]]])
deg_null = np.concatenate(parts)





In [11]:
null_curves

,x_kind,bin,observed_mean,observed_ci_low,observed_ci_high,null_mean,null_lo,null_hi
0,count,0,0.027697,0.026960,0.028427,0.119871,0.119767,0.119988
1,count,1,0.104031,0.102277,0.105784,0.146460,0.145370,0.147394
2,count,2,0.119035,0.116918,0.120970,0.145482,0.143969,0.146767
3,count,3,0.132435,0.129741,0.134927,0.142841,0.140429,0.144823
4,count,4,0.138232,0.135114,0.140759,0.140096,0.137705,0.142215
5,count,5,0.141313,0.137973,0.144456,0.137132,0.134585,0.139874
6,count,6,0.145109,0.141225,0.148658,0.135305,0.132269,0.138047
7,count,7,0.148342,0.143871,0.151864,0.133954,0.130386,0.137279
8,count,8,0.158567,0.152738,0.163314,0.133328,0.132862,0.133888
9,n_eff,0,0.027697,0.026960,0.028427,0.119871,0.119767,0.119988


In [ ]:
null_curves = pd.read_parquet(RUN_DIR / "null_curves" / f"null_curves_cluster_average_seed{SEED}.parquet")
null_curves.replace({'count': 's_raw', 'n_eff': 's_eff'}, inplace=True)
null_curves.rename(columns={"x_kind": "overlap_kind"}, inplace=True)

null_curves["observed_mean_cosine"] = null_curves["observed_mean"]
null_curves["observed_err_low"] = null_curves["observed_mean"] - null_curves["observed_ci_low"]
null_curves["observed_err_high"] = null_curves["observed_ci_high"] - null_curves["observed_mean"]

null_curves["null_mean_cosine"] = null_curves["null_mean"]
null_curves["null_err_low"] = null_curves["null_mean"] - null_curves["null_lo"]
null_curves["null_err_high"] = null_curves["null_hi"] - null_curves["null_mean"]

null_curves[["overlap_kind", "bin", "observed_mean_cosine", "observed_err_low", "observed_err_high", "null_mean_cosine", "null_err_low", "null_err_high"]].to_csv(FIGURE_DATA_DIR / f"similarity_vs_overlaps_hcm_null_seed{SEED}.csv", index=False)

   overlap_kind  bin  observed_mean  observed_ci_low  observed_ci_high  \
0         s_raw    0       0.027697         0.026960          0.028427   
1         s_raw    1       0.104031         0.102277          0.105784   
2         s_raw    2       0.119035         0.116918          0.120970   
3         s_raw    3       0.132435         0.129741          0.134927   
4         s_raw    4       0.138232         0.135114          0.140759   
5         s_raw    5       0.141313         0.137973          0.144456   
6         s_raw    6       0.145109         0.141225          0.148658   
7         s_raw    7       0.148342         0.143871          0.151864   
8         s_raw    8       0.158567         0.152738          0.163314   
9         s_eff    0       0.027697         0.026960          0.028427   
10        s_eff    1       0.105443         0.103935          0.107188   
11        s_eff    2       0.126368         0.123701          0.128612   
12        s_eff    3       0.138859   